In [ ]:
import gradio as gr
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API key
load_dotenv()

# Create OpenAI client

# BASE_URL = "https://openrouter.ai/api/v1"
# api_key = os.getenv('OPENROUTER_API_KEY')
# MODEL = 'openai/gpt-5-nano'

BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv('GOOGLE_API_KEY')
MODEL = 'gemini-2.5-flash'
myAI = OpenAI(base_url=BASE_URL, api_key=api_key)

# -----------------------------
# OpenAI Function
# -----------------------------
def ask_ai(prompt):
    if not prompt.strip():
        return "Please enter a prompt."

    stream = myAI.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"user","content":prompt}
        ],
        stream=True
    )
    response = ""
    for chunk in stream:
        token = chunk.choices[0].delta.content or ""
        response += token
        yield response

# -----------------------------
# Stop Button Function
# -----------------------------
def stop_generation():
    return "⛔ Generation stopped (demo)."

# -----------------------------
# Gradio UI
# -----------------------------
with gr.Blocks(title="OpenAI Chatbot") as demo:

    gr.Markdown("# 🤖 OpenAI Chatbot")

    prompt = gr.Textbox(
        label="Your Prompt",
        lines=5,
        placeholder="Ask anything..."
    )

    answer = gr.Textbox(
        label="AI Response",
        lines=10
    )

    with gr.Row():
        ask_btn = gr.Button("🚀 Ask AI", variant="primary")
        clear_btn = gr.Button("🗑️ Clear", variant="secondary")
        stop_btn = gr.Button("⛔ Stop", variant="stop")

    # Button Events
    ask_btn.click(
        fn=ask_ai,
        inputs=prompt,
        outputs=answer
    )

    clear_btn.click(
        fn=lambda: ("", ""),
        inputs=[],
        outputs=[prompt, answer]
    )

    stop_btn.click(
        fn=stop_generation,
        inputs=[],
        outputs=answer
    )

demo.launch()